<div dir="rtl">

# ניסוי השוואה: אימון e5-large בלי סינון של שליליים שקריים

המחברת זהה לאימון של סבב 2, חוץ מדבר אחד: **בחירת השליליים הקשים בלי שום סינון**. מוותרים רק על הפסקה הנכונה עצמה, וכל שאר הפסקאות הכי דומות יכולות לשמש כשליליות, כולל פסקאות מאותו ערך ופסקאות שדורגו מעל הפסקה הנכונה.

כך בודקים אם הסינון באמת עוזר. כל שאר ההגדרות זהות: אותו מודל, אותו זרע אקראי, אותם פרמטרים.

**הרצה ב-Kaggle:** בהגדרות המחברת (צד ימין) בחר `Accelerator: GPU T4 x2` או `GPU P100`, והפעל `Internet: On`. אחר כך `Save Version → Save & Run All (Commit)`. ההרצה ממשיכה גם אם סוגרים את הדפדפן, והתוצאה נשמרת בקובץ `ablation_results.csv`.

</div>

<div dir="rtl">

## 1. התקנה והגדרות

</div>

In [ ]:
!pip install -q -U sentence-transformers rank_bm25

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use a single GPU even on "T4 x2"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less GPU memory fragmentation
import json, random, re, time, gc
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")

LARGE = "intfloat/multilingual-e5-large"
FT_L = "Fine-tuned e5-large, no filtering (ablation)"
FT_L_NAME = "hebrew-qa-retriever-e5-large"
DATASET_NAME = "hebrew-qa-retrieval-data"
EPOCHS_L, BATCH_L, MINI_L, LR_L, N_NEG, MAX_LEN = 1, 64, 8, 1e-5, 2, 384

In [ ]:
from huggingface_hub import hf_hub_download
data_repo = f"ElyKok/{DATASET_NAME}"   # public dataset with the earlier results, no login needed

<div dir="rtl">

## 2. טעינת HeQ

</div>

In [ ]:
!git clone -q --depth 1 https://github.com/NNLP-IL/Hebrew-Question-Answering-Dataset heq 2>/dev/null || echo "already cloned"

corpus, para_id = [], {}
queries = {"train": [], "val": [], "test": []}
for split, fname in [("train", "train v1.1.json"), ("val", "val v1.1.json"), ("test", "test v1.1.json")]:
    data = json.load(open(f"heq/data/data v1.1/{fname}", encoding="utf-8"))["data"]
    for article in data:
        for p in article["paragraphs"]:
            text = p["context"].strip()
            if text not in para_id:
                para_id[text] = len(corpus)
                corpus.append({"id": len(corpus), "title": article["title"],
                               "source": article["source"], "split": split, "text": text})
            pid = para_id[text]
            for q in p["qas"]:
                if not q["is_impossible"]:
                    queries[split].append({"qid": q["id"], "question": q["question"].strip(),
                                           "pid": pid, "source": article["source"]})
corpus_texts = [p["text"] for p in corpus]
test_q = queries["test"]
pos = [q["pid"] for q in test_q]
print(f"Paragraphs: {len(corpus):,} | " + " | ".join(f"{s}: {len(v):,}" for s, v in queries.items()))

<div dir="rtl">

## 3. מדידה ותוצאות הסבב הקודם

אותה פונקציית מדידה כמו במחברת הראשונה. הטבלה מתחילה מהתוצאות שנשמרו ב-Hugging Face, ושורות חדשות מתווספות אליה.

</div>

In [ ]:
def metrics_from_ranks(ranks):
    r = np.asarray(ranks)
    return {"R@1": np.mean(r <= 1) * 100, "R@5": np.mean(r <= 5) * 100, "R@10": np.mean(r <= 10) * 100,
            "MRR@10": np.mean(np.where(r <= 10, 1 / r, 0)) * 100,
            "NDCG@10": np.mean(np.where(r <= 10, 1 / np.log2(r + 1), 0)) * 100}

def ranks_from_scores(scores, pos):
    pos = torch.as_tensor(pos, device=scores.device)
    pos_scores = scores[torch.arange(len(pos), device=scores.device), pos]
    return ((scores > pos_scores[:, None]).sum(1) + 1).cpu().numpy()

all_ranks = {}
all_results = pd.read_csv(hf_hub_download(data_repo, "results.csv", repo_type="dataset")).to_dict("records")

def report(name, ranks, qs):
    all_ranks[name] = ranks
    row = {"Model": name, **metrics_from_ranks(ranks)}
    for src in ["Wikipedia", "Geektime"]:
        mask = np.array([q["source"] == src for q in qs])
        row[f"R@5 {src}"] = metrics_from_ranks(ranks[mask])["R@5"]
    all_results[:] = [r for r in all_results if r["Model"] != name] + [row]
    display(pd.DataFrame(all_results).set_index("Model").round(1))

def encode(model, texts, kwargs, bs=64):
    return model.encode(texts, batch_size=bs, convert_to_tensor=True, normalize_embeddings=True,
                        show_progress_bar=True, **kwargs)

def free_gpu():
    gc.collect(); torch.cuda.empty_cache()

display(pd.DataFrame(all_results).set_index("Model").round(1))

<div dir="rtl">

## 4. e5-large בלי אימון

הווקטורים של e5-large משמשים לבחירת השליליים הקשים, ולהשוואה.

</div>

In [ ]:
E5 = dict(q=dict(prompt="query: "), d=dict(prompt="passage: "))
m = SentenceTransformer(LARGE, device=device).half()
m.max_seq_length = 512
C_large = encode(m, corpus_texts, E5["d"]).float().cpu()
Q_large_test = encode(m, [q["question"] for q in test_q], E5["q"]).float().cpu()
Q_large_train = encode(m, [q["question"] for q in queries["train"]], E5["q"]).float().cpu()
del m; free_gpu()
report("multilingual-e5-large", ranks_from_scores(Q_large_test @ C_large.T, pos), test_q)

<div dir="rtl">

## 5. שליליים קשים

לכל שאלת אימון, e5-large מחפש את הפסקאות הכי דומות, רק מתוך פסקאות האימון. **בלי סינון:** מוותרים רק על הפסקה הנכונה עצמה, ובוחרים באקראי 2 מתוך 15 הפסקאות הכי דומות.

</div>

In [ ]:
train_pids = [p["id"] for p in corpus if p["split"] == "train"]
train_index = {pid: j for j, pid in enumerate(train_pids)}
C_train = C_large[train_pids].to(device)
scores = Q_large_train.to(device) @ C_train.T
top_scores, top_idx = [t.cpu() for t in scores.topk(50, dim=1)]
del C_train, scores; free_gpu()

train_rows = []
for i, q in enumerate(tqdm(queries["train"], desc="Mining hard negatives")):
    pool = [train_pids[j] for j in top_idx[i].tolist() if train_pids[j] != q["pid"]][:15]   # no filtering
    row = {"anchor": "query: " + q["question"], "positive": "passage: " + corpus[q["pid"]]["text"]}
    for n, pid in enumerate(random.sample(pool, N_NEG), 1):
        row[f"negative_{n}"] = "passage: " + corpus[pid]["text"]
    train_rows.append(row)
print(f"Training rows: {len(train_rows):,}")

<div dir="rtl">

## 6. אימון

כמו בסבב הראשון: `CachedMultipleNegativesRankingLoss`, בלי כפילויות ב-batch, ומדידה על ה-validation כל 100 צעדים. נקודת ההתחלה של e5-large על ה-validation היא Ndcg@10 של כ-0.806.

לוקח בערך שעה.

</div>

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sentence_transformers.evaluation import InformationRetrievalEvaluator

val_evaluator = InformationRetrievalEvaluator(
    queries={q["qid"]: "query: " + q["question"] for q in queries["val"]},
    corpus={str(p["id"]): "passage: " + p["text"] for p in corpus if p["split"] != "test"},
    relevant_docs={q["qid"]: {str(q["pid"])} for q in queries["val"]},
    name="heq-val", mrr_at_k=[10], ndcg_at_k=[10], accuracy_at_k=[1, 5, 10], precision_recall_at_k=[10],
    show_progress_bar=False)

free_gpu()
print(f"GPU memory in use before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
model = SentenceTransformer(LARGE, device=device)
model.max_seq_length = MAX_LEN
args = SentenceTransformerTrainingArguments(
    output_dir="checkpoints-large", num_train_epochs=EPOCHS_L, per_device_train_batch_size=BATCH_L,
    learning_rate=LR_L, warmup_ratio=0.1, fp16=True, batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="steps", eval_steps=100, save_strategy="no", logging_steps=20,
    report_to="none", seed=SEED)
trainer = SentenceTransformerTrainer(
    model=model, args=args,
    train_dataset=Dataset.from_list(train_rows).shuffle(seed=SEED),
    loss=CachedMultipleNegativesRankingLoss(model, mini_batch_size=MINI_L),
    evaluator=val_evaluator)
start = time.time()
trainer.train()
print(f"Training took {(time.time() - start) / 60:.1f} minutes")
model.save("finetuned-large")

<div dir="rtl">

## 7. תוצאה

השורה החדשה בטבלה היא תוצאת הניסוי. משווים אותה לשורה של e5-large המאומן עם סינון, מסבב 2.

</div>

In [ ]:
del trainer; free_gpu()
model.half()
model.max_seq_length = 512
C_ft = encode(model, corpus_texts, E5["d"]).float()
Q_ft = encode(model, [q["question"] for q in test_q], E5["q"]).float()
report(FT_L, ranks_from_scores(Q_ft @ C_ft.T, pos), test_q)
pd.DataFrame([r for r in all_results if r["Model"] == FT_L]).set_index("Model").round(1).to_csv("ablation_results.csv")
print("Saved ablation_results.csv")

In [ ]:
base_r, ft_r = all_ranks["multilingual-e5-large"], all_ranks[FT_L]
improved = np.where((ft_r == 1) & (base_r > 1))[0]
worse = np.where((ft_r > 1) & (base_r == 1))[0]
print(f"e5-large -> fine-tuned: improved to rank 1: {len(improved)} | dropped from rank 1: {len(worse)}\n")
for idx, title in [(improved, "Improved"), (worse, "Got worse")]:
    print(f"=== {title} ===")
    for i in idx[:5]:
        q = test_q[i]
        print(f"Q: {q['question']}\n   base rank: {base_r[i]}  ->  fine-tuned rank: {ft_r[i]}")
        print(f"   paragraph: {corpus[q['pid']]['text'][:150]}...\n")